In [ ]:
import pandas as pd
from energy_model.evaluation.model_evaluator import ModelEvaluator
from sklearn.model_selection import train_test_split
from energy_model.configs.columns import ProcessColumns
from energy_model.pipelines.pipeline_utils import extract_x_y
from sklearn.ensemble import HistGradientBoostingRegressor


In [ ]:
process_real_time_df = pd.read_csv(
    r"C:\Users\Administrator\Desktop\GreenSecurityAll\documents\gt\with batch 10 minutes\basic - system based - process of interest\all_durations\system_process_df_new.csv", index_col=0)

system_real_time_df = pd.read_csv(
    r"C:\Users\Administrator\Desktop\GreenSecurityAll\documents\gt\with batch 10 minutes\basic - system based - process of interest\all_durations\system_only_df_new.csv", index_col=0)

process_long_term_df = pd.read_csv(
    r"C:\Users\Administrator\Desktop\GreenSecurityAll\documents\gt\with batch 10 minutes\aggregated - system based - process of interest\system_process_df.csv")

system_long_term_df = pd.read_csv(
    r"C:\Users\Administrator\Desktop\GreenSecurityAll\documents\gt\with batch 10 minutes\aggregated - system based - process of interest\system_only_df.csv")

In [ ]:
PROCESS_REAL_TIME_MODEL = HistGradientBoostingRegressor
PROCESS_REAL_TIME_MODEL_PARAMETERS = {"max_depth": 8,
                                      "l2_regularization": 0.3,
                                      "max_iter": 400,
                                      "quantile": 0.5}

PROCESS_LONG_TERM_MODEL = HistGradientBoostingRegressor
PROCESS_LONG_TERM_MODEL_PARAMETERS = {"max_depth": 8,
                                      "l2_regularization": 1.0,
                                      "loss": "quantile",
                                      "max_iter": 600,
                                      "quantile": 0.7}

SYSTEM_REAL_TIME_MODEL = HistGradientBoostingRegressor
SYSTEM_REAL_TIME_MODEL_PARAMETERS = {"max_depth": 5,
                                     "l2_regularization": 0.3,
                                     "max_iter": 800,
                                     "quantile": 0.5}

SYSTEM_LONG_TERM_MODEL = HistGradientBoostingRegressor
SYSTEM_LONG_TERM_MODEL_PARAMETERS = {"max_depth": 8,
                                     "l2_regularization": 1.0,
                                     "loss": "quantile",
                                     "max_iter": 600,
                                     "quantile": 0.7}

In [ ]:
cpu_thresholds = [0.10, 0.25, 0.50, 0.75, 0.90]

In [ ]:

def get_cpu_thresholds_results(df: pd.DataFrame, model_class, model_parameters):
    for cpu_threshold in cpu_thresholds:
        print(f"Calculating for CPU >= {cpu_threshold}")
        df_cpu = df[df[ProcessColumns.CPU_PROCESS_COL] >= cpu_threshold]
        if df_cpu.empty:
            print("No data found")
            continue
        else:
            print("Numbers of samples: ", len(df_cpu))
        model = model_class(**model_parameters)

        x, y = extract_x_y(df_cpu, target_column=ProcessColumns.ENERGY_USAGE_PROCESS_COL)
        x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2)
        model.fit(x_train, y_train)
        y_pred = model.predict(x_test)
        ev = ModelEvaluator()
        res = ev.evaluate(y_test, y_pred)
        ev.print_results(res)


# Real Time Process Model

In [ ]:
process_real_time_df

In [ ]:
get_cpu_thresholds_results(process_real_time_df, PROCESS_REAL_TIME_MODEL, PROCESS_REAL_TIME_MODEL_PARAMETERS)

# Real Time System Model

In [ ]:
get_cpu_thresholds_results(system_real_time_df, SYSTEM_REAL_TIME_MODEL, SYSTEM_REAL_TIME_MODEL_PARAMETERS)

# Long Term Process Model

In [ ]:
get_cpu_thresholds_results(process_long_term_df, PROCESS_LONG_TERM_MODEL, PROCESS_LONG_TERM_MODEL_PARAMETERS)

# Long Term System Model

In [ ]:
get_cpu_thresholds_results(system_long_term_df, SYSTEM_LONG_TERM_MODEL, SYSTEM_LONG_TERM_MODEL_PARAMETERS)

# Paired t-test

In [ ]:
from sklearn.metrics import mean_squared_error
import math
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.base import clone
from energy_model.configs.columns import ProcessColumns
from sklearn.metrics import mean_squared_error

def compute_rrmse(y_true, y_pred):
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    return rmse / np.mean(y_true) * 100

In [ ]:
def evaluate_thresholds_cv(
    df, model_class, model_config, target_col, cpu_col=ProcessColumns.CPU_PROCESS_COL,
    thresholds=[0.1, 0.25, 0.5, 0.75, 0.9], n_splits=5, random_state=42):
    """
    Runs 5-fold CV for each CPU threshold and computes:
    - Mean RRMSE
    - Std RRMSE
    - Per-fold RRMSE values

    Returns:
        results_df (summary table)
        detailed_results (dictionary with fold values)
    """

    results = []
    detailed_results = {}

    for threshold in thresholds:

        # Filter dataset
        df_threshold = df[df[cpu_col] >= threshold].copy()
        if len(df_threshold) < n_splits:
            print(f"Skipping threshold {threshold}: not enough samples.")
            continue

        X = df_threshold.drop(columns=[target_col])
        y = df_threshold[target_col]

        kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)

        fold_rrmses = []
        for train_idx, test_idx in kf.split(X):

            X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
            y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

            model = model_class(**model_config)
            model.fit(X_train, y_train)

            y_pred = model.predict(X_test)

            rrmse = compute_rrmse(y_test.values, y_pred)
            fold_rrmses.append(rrmse)

        mean_rrmse = np.mean(fold_rrmses)
        std_rrmse = np.std(fold_rrmses, ddof=1)
        results.append({
            "threshold": threshold,
            "num_samples": len(df_threshold),
            "mean_rrmse": mean_rrmse,
            "std_rrmse": std_rrmse
        })

        detailed_results[threshold] = fold_rrmses

    results_df = pd.DataFrame(results)

    return results_df, detailed_results

In [ ]:
df_process = pd.read_csv(
    r"C:\Users\Administrator\Desktop\GreenSecurityAll\documents\gt\with batch 10 minutes\basic - system based - process of interest\all_durations\system_process_df_new.csv", index_col=0)
process_model_config = {"max_depth": 8,
                                      "l2_regularization": 0.3,
                                      "max_iter": 400,
                                      "quantile": 0.5}

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor


results_df, fold_values = evaluate_thresholds_cv(
    df=df_process,
    model_class=HistGradientBoostingRegressor,
    model_config=process_model_config,
    target_col=ProcessColumns.ENERGY_USAGE_PROCESS_COL
)


In [ ]:

print(results_df)

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import t

def compute_confidence_intervals(results_df, n_folds=5, alpha=0.05):
    """
    Adds 95% confidence intervals to results_df.
    """

    df = results_df.copy()
    t_val = t.ppf(1 - alpha/2, df=n_folds - 1)

    margins = t_val * df["std_rrmse"] / np.sqrt(n_folds)

    df["ci_lower"] = df["mean_rrmse"] - margins
    df["ci_upper"] = df["mean_rrmse"] + margins

    return df

In [ ]:
results_with_ci = compute_confidence_intervals(results_df)

In [ ]:
print(results_with_ci)

In [ ]:
from scipy.stats import ttest_rel
from statsmodels.stats.multitest import multipletests

def paired_tests_against_best(fold_values, best_threshold=0.5, alpha=0.05):
    """
    Performs paired t-tests comparing best_threshold against others.
    Applies Holm correction.
    """

    best_scores = np.array(fold_values[best_threshold])

    comparisons = []
    p_values = []

    for threshold, scores in fold_values.items():

        if threshold == best_threshold:
            continue

        scores = np.array(scores)

        # One-sided paired t-test (best < other)
        t_stat, p_two_sided = ttest_rel(best_scores, scores)

        # Convert to one-sided
        if t_stat < 0:
            p_one_sided = p_two_sided / 2
        else:
            p_one_sided = 1 - (p_two_sided / 2)

        comparisons.append(threshold)
        p_values.append(p_one_sided)

    # Holm correction
    reject, p_corrected, _, _ = multipletests(
        p_values, alpha=alpha, method="holm"
    )

    results = pd.DataFrame({
        "compared_threshold": comparisons,
        "p_value_raw": p_values,
        "p_value_corrected": p_corrected,
        "significant": reject
    })

    return results

In [ ]:
test_results = paired_tests_against_best(fold_values, best_threshold=0.5)

In [ ]:
print(test_results)

In [ ]:
test_results = paired_tests_against_best(fold_values, best_threshold=0.9)

In [ ]:
print(test_results)